# Notebook 03 — Model Training & Evaluation
**Project:** Medical Insurance Cost Prediction  
**Goal:** Train 5 regression models, compare performance, identify the best model.

---
### Models Evaluated
1. Linear Regression (baseline)
2. Ridge Regression (L2 regularization)
3. Lasso Regression (L1 regularization / feature selection)
4. Random Forest (ensemble, non-linear)
5. XGBoost (gradient boosting)

### Metrics Used
- **R²** — Proportion of variance explained (higher = better)
- **RMSE** — Root Mean Squared Error in dollars (lower = better)
- **MAE** — Mean Absolute Error in dollars (lower = better)
- **CV R²** — 5-fold cross-validated R² (checks for overfitting)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib

from src.data_loader import load_raw_data
from src.preprocessing import preprocess
from src.config import TARGET_COLUMN, TEST_SIZE, RANDOM_STATE, MODELS_DIR

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Load processed data
df = preprocess(load_raw_data(), save=False)
print('Dataset shape:', df.shape)
df.head()

## 1. Train/Test Split

In [ ]:
drop_cols = [TARGET_COLUMN, 'bmi_category', 'age_group']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Features         : {len(feature_cols)}')
print(f'\nFeature list: {feature_cols}')

## 2. Train All Models

In [ ]:
models = {
    'Linear Regression' : LinearRegression(),
    'Ridge Regression'  : Ridge(alpha=1.0),
    'Lasso Regression'  : Lasso(alpha=1.0),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'XGBoost'           : XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE, verbosity=0),
}

results = []
trained_models = {}

print(f'{"Model":<22} {"R2":>8} {"RMSE":>10} {"MAE":>10} {"CV R2":>8}')
print('-' * 62)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    cv   = cross_val_score(model, X, y, cv=5, scoring='r2').mean()

    results.append({'Model': name, 'R2_Score': round(r2,4), 'RMSE': round(rmse,2),
                    'MAE': round(mae,2), 'CV_R2_Mean': round(cv,4)})
    trained_models[name] = model

    # Save model
    fname = name.lower().replace(' ', '_') + '.pkl'
    joblib.dump(model, MODELS_DIR / fname)

    print(f'{name:<22} {r2:>8.4f} {rmse:>10,.2f} {mae:>10,.2f} {cv:>8.4f}')

results_df = pd.DataFrame(results).sort_values('R2_Score', ascending=False)
print('\nAll models saved to models/')

## 3. Model Comparison

In [ ]:
results_df.style.background_gradient(subset=['R2_Score'], cmap='Greens') \
               .background_gradient(subset=['RMSE', 'MAE'], cmap='Reds_r') \
               .format({'R2_Score': '{:.4f}', 'RMSE': '${:,.2f}', 'MAE': '${:,.2f}', 'CV_R2_Mean': '{:.4f}'})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['R2_Score', 'RMSE', 'MAE']
colors  = ['steelblue', 'tomato', 'seagreen']
ascending = [False, True, True]

for ax, metric, color, asc in zip(axes, metrics, colors, ascending):
    data = results_df.sort_values(metric, ascending=asc)
    bars = ax.barh(data['Model'], data[metric], color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{metric}', fontweight='bold')
    ax.set_xlabel(metric)
    # Add value labels
    for bar, val in zip(bars, data[metric]):
        ax.text(bar.get_width() * 0.98, bar.get_y() + bar.get_height()/2,
                f'{val:,.4f}' if metric == 'R2_Score' else f'${val:,.0f}',
                va='center', ha='right', color='white', fontsize=9, fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/09_model_comparison.png', bbox_inches='tight')
plt.show()

## 4. Actual vs Predicted — Best Model

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = trained_models[best_name]
y_pred_best = best_model.predict(X_test)
r2_best = r2_score(y_test, y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_best, alpha=0.5, color='steelblue', s=25)
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')
axes[0].set_xlabel('Actual Charges ($)')
axes[0].set_ylabel('Predicted Charges ($)')
axes[0].set_title(f'{best_name} — Actual vs Predicted (R²={r2_best:.4f})')
axes[0].legend()

# Residuals
residuals = y_test - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.5, color='seagreen', s=25)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted Charges ($)')
axes[1].set_ylabel('Residuals ($)')
axes[1].set_title(f'{best_name} — Residual Plot')

plt.suptitle(f'Best Model: {best_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/10_actual_vs_predicted.png', bbox_inches='tight')
plt.show()

## 5. Feature Importance (Random Forest)

In [ ]:
rf_model = trained_models['Random Forest']
importance = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values()

plt.figure(figsize=(9, 6))
colors_imp = ['tomato' if i >= len(importance) - 3 else 'steelblue' for i in range(len(importance))]
importance.plot(kind='barh', color=colors_imp, edgecolor='white')
plt.title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../reports/figures/11_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 5 most important features:')
print(importance.sort_values(ascending=False).head())

## 6. Save Results

In [ ]:
results_df.to_csv('../reports/model_results.csv', index=False)
print('Results saved to reports/model_results.csv')
print('\nFinal Rankings:')
print(results_df.to_string(index=False))

## 7. Conclusion

| Rank | Model | R² | RMSE | Verdict |
|---|---|---|---|---|
| 1 | Linear Regression | ~0.907 | ~$4,142 | Best R² & simplest — recommended for deployment |
| 2 | Ridge Regression | ~0.906 | ~$4,151 | Slight regularization benefit |
| 3 | Lasso Regression | ~0.907 | ~$4,144 | Same as Linear (no irrelevant features to zero out) |
| 4 | XGBoost | ~0.891 | ~$4,474 | Good but lower R² on this dataset |
| 5 | Random Forest | ~0.880 | ~$4,698 | Strong CV score but overfits slightly |

**Key Takeaways:**
- `smoker` is the dominant feature — far outweighs all others
- `smoker_obese` interaction captures the highest-cost segment
- Linear Regression performs surprisingly well, confirming the largely linear relationships found in EDA
- All models achieve R² > 0.88, indicating the features explain most of the variance in insurance charges